In [31]:
from experiments import (
    load_selfplay_results,
    load_tournament_results
)
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
from pathlib import Path

In [32]:
from pathlib import Path


scores_0 = load_selfplay_results (Path("../../../results/other/memory_n_ql/pd"), 0.00)
scores_05 = load_selfplay_results (Path("../../../results/other/memory_n_ql/pd"), 0.05)
scores_1 = load_selfplay_results (Path("../../../results/other/memory_n_ql/pd"), 0.10)
# Filter to only include rows where Player index equals Opponent index (self-play)
scores_0 = scores_0[scores_0['Player index'] == scores_0['Opponent index']]
scores_05 = scores_05[scores_05['Player index'] == scores_05['Opponent index']]
scores_1 = scores_1[scores_1['Player index'] == scores_1['Opponent index']]
# combine with noise level
scores_0['noise_level'] = 0.00
scores_05['noise_level'] = 0.05
scores_1['noise_level'] = 0.10

scores = pd.concat([scores_0, scores_05, scores_1])


In [33]:
# Aggregate score over interaction index (mean across interactions for each player pair and repetition)
scores_agg_interaction = scores.groupby(['Player index', 'Opponent index', 'repetition', 'noise_level'])['Score'].mean().reset_index()

# Then aggregate over repetition (mean and stdev across repetitions for each player pair)
scores_final = scores_agg_interaction.groupby(['Player index', 'Opponent index', 'noise_level'])['Score'].agg(['mean', 'std']).reset_index()

# Display the final aggregated scores
scores_final



,Player index,Opponent index,noise_level,mean,std
0,0,0,0.00,1890.05,290.072639
1,0,0,0.05,1947.50,141.087050
2,0,0,0.10,1998.00,92.657074
3,1,1,0.00,2482.90,146.888809
4,1,1,0.05,2458.35,106.626047
5,1,1,0.10,2453.95,99.199980
6,2,2,0.00,2721.95,65.220332
7,2,2,0.05,2686.80,53.922887
8,2,2,0.10,2646.20,56.470838
9,3,3,0.00,2764.10,38.104972


In [34]:
# drop index column
scores_final = scores_final.drop(columns=['Player index'])
# rename Opponent index to "Memory length"
scores_final = scores_final.rename(columns={'Opponent index': 'Memory length'})
# save to csv
scores_final.to_csv('scores_final.csv', index=False)

In [35]:
from scipy.stats import t

# Compute 95% confidence intervals using t-distribution
# Need sample size: from earlier grouping we have 5 repetitions
n_reps = 5
alpha = 0.05
t_critical = t.ppf(1 - alpha/2, df=n_reps - 1)  # two-tailed 95% CI

# Compute margin of error for each mean
scores_final['ci_half_width'] = t_critical * scores_final['std'] / np.sqrt(n_reps)

# Pivot the data to have memory length as index and noise levels as columns
pivot_mean = scores_final.pivot(index='Player index', columns='noise_level', values='mean')
pivot_ci = scores_final.pivot(index='Player index', columns='noise_level', values='ci_half_width')

# Plot mean score vs memory length for each noise level
plt.figure(figsize=(10, 6))

noise_levels = scores_final['noise_level'].unique()
for noise in noise_levels:
    if noise in pivot_mean.columns:
        plt.errorbar(
            x=pivot_mean.index + 1,  # +1 to convert from 0-based to 1-based memory length
            y=pivot_mean[noise],
            yerr=pivot_ci[noise],
            label=f'Noise = {noise}',
            marker='o',
            capsize=5
        )

plt.xlabel('Memory Length')
plt.ylabel('Mean Score')
plt.title('Mean Score vs Memory Length for Q-Learning Agents (95% CI)')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()


KeyError: 'Player index'